# 📊 Zone Response Analysis
**Purpose:** Analyse how zone temperature, humidity, and CO₂ respond to VAV flow commands.  
Select a single zone at the top and all downstream plots update automatically.

In [1]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from data_wrangling import DataInspector, DataPlotter

# Load data
CSV_PATH = './results/state_log.csv'
df = pd.read_csv(CSV_PATH)

# Build datetime index (EnergyPlus uses Hour=24 for midnight → roll to next day)
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)

ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']

print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')
# df.head(3)


Loaded 459 timesteps, columns: 127


In [2]:
ZONE = 'SPACE1-1'

## 1 · Temperature Response vs Flow Command

In [3]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Zone & Outdoor Temperature', 'VAV Flow Command'],
    vertical_spacing=0.08
)

# Row 1: Temperatures
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Out_Temp_C'],
    name='Outdoor T', line=dict(color='#636EFA', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Fan_Out_Temp_C'],
    name='Supply Air T', line=dict(color='#00CC96', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_Temp_C'],
    name=f'{ZONE} T_in', line=dict(color='#EF553B', width=2)), row=1, col=1)
# fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_T_m_C'],
#     name=f'{ZONE} T_m (Radiant)', line=dict(color='#FFA15A', width=1.5)), row=1, col=1)

# Row 2: VAV flow
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.update_layout(template='plotly_dark', height=550,
    title=f'Temperature Response — {ZONE}',
    legend=dict(orientation='h', y=-0.12))
fig.update_yaxes(title_text='°C', row=1, col=1)
fig.update_yaxes(title_text='kg/s', row=2, col=1)
fig.show()

/home/jazz/.local/lib/python3.14/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## 2 · Humidity Response vs Flow Command

In [4]:


fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Zone Humidity (RH & W_in)', 'VAV Flow Command'],
    specs=[[{'secondary_y': True}], [{}]],
    vertical_spacing=0.08
)


# RH on primary y-axis
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_RH_pct'],
    name=f'{ZONE} RH %', line=dict(color='#19D3F3', width=2)), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Out_RH_pct'],
    name='Outdoor RH %', line=dict(color='#636EFA', width=1, dash='dot')), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Fan_Out_RH_pct'],
    name='Supply RH %', line=dict(color='#00CC96', width=1, dash='dot')), row=1, col=1, secondary_y=False)

# Row 2: Flow
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.add_hrect(
    y0=30, 
    y1=60, 
    row=1, 
    col=1,
    secondary_y=False,
    fillcolor="#84C5B1", 
    opacity=0.15,
    line_width=0,
    annotation_text="Comfort Zone (30-60%)", 
    annotation_position="top left",
    annotation_font_color="white"
)
fig.update_layout(template='plotly_dark', height=550,
    title=f'Humidity Response — {ZONE}',
    legend=dict(orientation='h', y=-0.12))

# 2. Force Y-axis to 0-110 for RH
fig.update_yaxes(title_text='RH %', range=[0, 110], row=1, col=1, secondary_y=False)

fig.update_yaxes(title_text='kg/kg', row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text='kg/s', row=2, col=1)

fig.show()

## 3 · CO₂ Response vs Flow & Occupancy

In [5]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.45, 0.30, 0.25],
    subplot_titles=['Zone CO₂ Concentration', 'VAV Flow Command', 'Occupancy'],
    vertical_spacing=0.07
)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_CO2_ppm'],
    name=f'{ZONE} CO₂', line=dict(color='#B6E880', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Fan_Out_CO2_ppm'],
    name='Supply CO₂', line=dict(color='#00CC96', width=1, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Outdoor_Air_CO2_ppm'],
    name='Outside CO₂', line=dict(color='#AB63FA', width=1, dash='dot')), row=1, col=1)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_Occupants'],
    name='Occupants', fill='tozeroy',
    line=dict(color='#FF6692', width=1.5)), row=3, col=1)

fig.add_hrect(
    y0=0, 
    y1=1000, 
    row=1, 
    col=1,
    secondary_y=False,
    fillcolor="#84C5B1", 
    opacity=0.15,
    line_width=0,
    annotation_text="Comfort Zone (<1000ppm)", 
    annotation_position="top left",
    annotation_font_color="#84C5B1"
)
fig.update_yaxes(title_text='CO₂ (ppm)', range=[-50, 1500], row=1, col=1, secondary_y=False)

fig.update_layout(template='plotly_dark', height=650,
    title=f'CO₂ Response — {ZONE}',
    legend=dict(orientation='h', y=-0.08))
fig.update_yaxes(title_text='ppm', row=1, col=1)
fig.update_yaxes(title_text='kg/s', row=2, col=1)
fig.update_yaxes(title_text='Count', row=3, col=1)
fig.show()

## 4 · Supply Air Conditions (Outdoor → Fan Out)

In [6]:
fig = make_subplots(
    rows=2, cols=2, shared_xaxes=True,
    subplot_titles=['AHU Temperature Path', 'AHU RH Path',
                    'AHU Flow Path', 'AHU CO₂ Path'],
    vertical_spacing=0.12, horizontal_spacing=0.08
)

nodes = ['Outdoor_Air', 'Mixed_Air', 'CC_Out', 'HC_Out', 'Fan_Out']
colors_ahu = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA']

for i, node in enumerate(nodes):
    c = colors_ahu[i]
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_Temp_C'],
        name=f'{node} T', line=dict(color=c, width=1.5), legendgroup=node), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_RH_pct'],
        name=f'{node} RH', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_Flow_kg_s'],
        name=f'{node} Flow', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_CO2_ppm'],
        name=f'{node} CO₂', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=2, col=2)

fig.update_layout(template='plotly_dark', height=600,
    title='Air Handling Unit — State Through Nodes',
    legend=dict(orientation='h', y=-0.08))
fig.show()

## 5 · Statistical Summary (data_wrangling)

In [40]:
# Use DataInspector from the data_wrangling library
zone_cols = [f'{ZONE}_Temp_C', f'{ZONE}_RH_pct', f'{ZONE}_CO2_ppm',
             f'{ZONE}_VAV_Flow_kg_s', f'{ZONE}_Occupants', f'{ZONE}_EquipLoad_W',
             'Out_Temp_C', 'Out_RH_pct']

di = DataInspector()
di.df = df[zone_cols].copy()
sum_df= di.get_summary(detailed=True)

--- Data Summary (Detailed) ---


,Column Name,Data Type,Total Records,Missing Values,Min,Max,Mean,Std Dev,Unique Values,Most Frequent
0,SPACE1-1_Temp_C,float64,2880,0,18.77,34.5800,21.291552,0.677969,NaN,NaN
1,SPACE1-1_RH_pct,float64,2880,0,17.47,78.1800,55.036188,7.871563,NaN,NaN
2,SPACE1-1_CO2_ppm,float64,2880,0,452.35,933.8300,516.078319,73.902255,NaN,NaN
3,SPACE1-1_VAV_Flow_kg_s,float64,2880,0,0.00,2.3518,0.463483,0.337490,NaN,NaN
4,SPACE1-1_Occupants,float64,2880,0,0.00,38.0000,1.993056,6.019531,NaN,NaN
5,SPACE1-1_EquipLoad_W,float64,2880,0,0.00,33427.7600,1029.736347,2434.567420,NaN,NaN
6,Out_Temp_C,float64,2880,0,11.70,35.0000,24.102181,4.619783,NaN,NaN
7,Out_RH_pct,float64,2880,0,36.00,100.0000,73.961458,16.182089,NaN,NaN


In [41]:
sum_df= di.summary_plot(
    columns=[f'{ZONE}_Temp_C',f'{ZONE}_RH_pct', f'{ZONE}_CO2_ppm',f'{ZONE}_VAV_Flow_kg_s', f'{ZONE}_Occupants'],
    numeric_plots=['violin', 'scatter', 'distplot'],
    separate_plots=True
    )

In [42]:
# Flow vs Temperature scatter using DataPlotter
DataPlotter.scatter(
    df, x_col=f'{ZONE}_VAV_Flow_kg_s', y_col=f'{ZONE}_Temp_C',
    color_col=f'{ZONE}_Occupants',
    title=f'Flow vs Temperature — {ZONE}'
).show()

In [43]:
import plotly.graph_objects as go
import numpy as np

# 1. Extract the data (dropping NaNs to prevent histogram math errors)
x_data = df[f'{ZONE}_RH_pct'].dropna()
y_data = df[f'{ZONE}_CO2_ppm'].dropna()

# 2. Compute the 2D histogram (Z-axis = frequency/density of points)
# You can increase 'bins' (e.g., 50 or 60) for a sharper surface, or decrease for smoother aggregation
H, xedges, yedges = np.histogram2d(x_data, y_data, bins=40)

# Calculate the centers of the bins to map the surface correctly
x_centers = (xedges[:-1] + xedges[1:]) / 2
y_centers = (yedges[:-1] + yedges[1:]) / 2

fig = go.Figure()

# 3. Add the 3D Surface for the data distribution
# H must be transposed (.T) so X and Y align correctly in Plotly
fig.add_trace(go.Surface(
    z=H.T, 
    x=x_centers, 
    y=y_centers,
    colorscale='Viridis',  # 'Plasma' or 'Inferno' also look great in dark mode
    name='Data Density',
    colorbar_title='Point Count',
    opacity=0.95
))

# 4. Draw the Comfort Zone 3D Wireframe Box
z_max = H.max()

# Define the coordinates for a 3D bounding box
# We use np.nan to "lift the pen" and draw the remaining vertical pillars cleanly
box_x = [
    30, 60, 60, 30, 30,      # Bottom loop
    30, 60, 60, 30, 30,      # Top loop
    np.nan, 60, 60,          # Pillar at 60 RH, 0 CO2
    np.nan, 60, 60,          # Pillar at 60 RH, 1000 CO2
    np.nan, 30, 30           # Pillar at 30 RH, 1000 CO2
]
box_y = [
    0, 0, 1000, 1000, 0,     # Bottom loop
    0, 0, 1000, 1000, 0,     # Top loop
    np.nan, 0, 0,            # Pillar at 60 RH, 0 CO2
    np.nan, 1000, 1000,      # Pillar at 60 RH, 1000 CO2
    np.nan, 1000, 1000       # Pillar at 30 RH, 1000 CO2
]
box_z = [
    0, 0, 0, 0, 0,           # Bottom loop (Z=0)
    z_max, z_max, z_max, z_max, z_max,  # Top loop (Z=z_max)
    np.nan, 0, z_max,        # Pillar at 60 RH, 0 CO2
    np.nan, 0, z_max,        # Pillar at 60 RH, 1000 CO2
    np.nan, 0, z_max         # Pillar at 30 RH, 1000 CO2
]

fig.add_trace(go.Scatter3d(
    x=box_x, y=box_y, z=box_z,
    mode='lines',
    line=dict(color='#00CC96', width=5), # Matching your supply line color theme
    name='Comfort Box (RH:30-60%, CO₂<1000)',
    showlegend=True
))

# 5. Update Layout and Styling
fig.update_layout(
    title=f'Bivariate State Space Density (RH vs CO₂) — {ZONE}',
    template='plotly_dark',
    height=800,
    scene=dict(
        xaxis_title='Relative Humidity (%)',
        yaxis_title='CO₂ (ppm)',
        zaxis_title='Frequency (Data Points)',
        xaxis=dict(range=[0, 100]), # Lock RH to standard 0-100 scale
        yaxis=dict(range=[0, max(2000, y_data.max())]), # Scale CO2 dynamically but at least up to 2000
    ),
    legend=dict(
        orientation='h', 
        y=-0.1,
        x=0.5,
        xanchor='center'
    )
)

fig.show()

In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']

# 1. Setup the Subplot Grid
cols = 3
rows = math.ceil(len(ZONES) / cols)

fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=[f'{zone}' for zone in ZONES],
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

# Custom colorscale
custom_colorscale = [
    [0.0, "black"],
    [0.1, "darkblue"],
    [0.5, "royalblue"],
    [1.0, "yellow"]
]

# 2. Loop through all zones
for i, zone in enumerate(ZONES):
    row = (i // cols) + 1
    col = (i % cols) + 1
    
    # Extract the data (ensure your DataFrame 'df' is loaded)
    x_data = df[f'{zone}_RH_pct'].dropna()
    y_data = df[f'{zone}_CO2_ppm'].dropna()

    # Add the 2D Histogram (Heatmap)
    fig.add_trace(go.Histogram2d(
        x=x_data,
        y=y_data,
        colorscale=custom_colorscale,
        showscale=(i == 0), 
        colorbar=dict(title='Data Points', x=1.02) if i == 0 else None,
        nbinsx=40,  
        nbinsy=40,
        zauto=True,
        xgap=1, # <--- Set to 1 for ALL zones to get the square grid look
        ygap=1  # <--- Set to 1 for ALL zones to get the square grid look
    ), row=row, col=col)

    # Draw the Comfort Zone Rectangle
    fig.add_shape(
        type="rect",
        x0=30, x1=60,
        y0=400, y1=1000,
        line=dict(color="#00CC96", width=2, dash='dash'),
        fillcolor="rgba(0, 204, 150, 0.05)",
        layer="above",
        row=row, col=col
    )

    # Add an annotation label for the comfort zone
    fig.add_annotation(
        x=60, 
        y=1000,
        text="Comfort Zone",
        showarrow=False,
        font=dict(color="#00CC96", size=10),
        xanchor="left",
        yanchor="bottom",
        xshift=4,
        yshift=4,
        row=row, col=col
    )

    # Hardcode Axes limits for consistency
    fig.update_xaxes(title_text='RH (%)', range=[0, 100], row=row, col=col)
    fig.update_yaxes(title_text='CO₂ (ppm)', range=[0, 1500], row=row, col=col)

# 3. Update Global Layout and Styling
fig.update_layout(
    title='Bivariate State Space Density (RH vs CO₂) — All Zones',
    template='plotly_dark',
    height=800,  
    width=1300,  
    plot_bgcolor='black', 
    paper_bgcolor='black'
)

fig.show()